## Passo a passo para rodar MPAS no container docker

git clone https://github.com/TempoHPC/MPAS-Model.git

In [ ]:
cd MPAS-Model

Precisa estar na branch: **branch_v8.2.2**

In [ ]:
git branch #verifica em qual branch está

In [ ]:
git chekout branch_v8.2.2 #muda para a branch branch_v8.2.2

Após isso, entrar no diretório onde estão os arquivos

In [ ]:
cd docker/nvhpc_24.9

In [ ]:
ls -ltr

Entre os arquivos, devem conter os seguintes:

In [ ]:
MPAS_v8.2.2.dockerfile              
run_mpas.sh

MPAS_v8.2.2.dockerfile

In [ ]:
# docker build --no-cache -t mpas:8.2.2 -f MPAS_v8.2.2.dockerfile .
# docker run --gpus all -it --entrypoint bash mpas:8.2.2
# docker run --gpus all -it --entrypoint bash --rm mpas:8.2.2
# docker exec -i -t <container_name> bash

FROM nvcr.io/nvidia/nvhpc:24.9-devel-cuda12.6-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive
SHELL ["/bin/bash", "-c"]


#Variáveis de diretórios principais

ENV MPAS_DIR=/home/monan/MPAS-Model_v8.2.2_tempohpc \
    BENCHMARK_DIR=/home/monan/MPAS-A_benchmark_120km_v7.0

#Instalar dependências do sistema
RUN apt update -y && apt upgrade -y && apt install -y \
    build-essential \
    curl \
    git \
    libbsd-dev \
    python3 \
    cmake \
    make \
    pkg-config \
    vim \
    environment-modules \
    m4 \
    perl \
    bzip2 \
    wget

#Criar usuário e home

RUN adduser --disabled-password --gecos "" monan
USER monan
WORKDIR /home/monan

#Baixar Spack

RUN wget https://github.com/spack/spack/releases/download/v0.23.1/spack-0.23.1.tar.gz && \
    tar zxvf spack-0.23.1.tar.gz

#Clonar MPAS
RUN git clone --single-branch --branch branch_v8.2.2 https://github.com/TempoHPC/MPAS-Model.git ${MPAS_DIR}


#Instalar Spack e compilar MPAS
RUN echo $USER && \
    echo $HOME && \
    cd && \
    source /usr/share/modules/init/bash && \
    module use /opt/nvidia/hpc_sdk/modulefiles && \
    module load nvhpc-openmpi3/24.9 && \
    source /home/monan/spack-0.23.1/share/spack/setup-env.sh && \
    spack compiler find && \
    spack external find m4 perl cmake openmpi bzip2 && \
    spack install parallelio%nvhpc@=24.9 ^parallel-netcdf ^netcdf-c@4.9.2~blosc~zstd && \
    export NETCDF=$(spack location -i netcdf-fortran) && \
    export PNETCDF=$(spack location -i parallel-netcdf) && \
    ln -sf $(spack location -i netcdf-c)/lib/libnetcdf* ${NETCDF}/lib/ && \
    cd ${MPAS_DIR} && \
    git pull && \
    make CORE=atmosphere clean && \
    make -j ${NUM_PROCS} pgi CORE=atmosphere USE_PIO=false OPENACC=true OPENMP=true PRECISION=single 2>&1 | tee make.output

#Baixar benchmark e extrair
RUN wget https://www2.mmm.ucar.edu/projects/mpas/benchmark/v7.0/MPAS-A_benchmark_120km_v7.0.tar.gz && \
    tar -xvzf MPAS-A_benchmark_120km_v7.0.tar.gz


#Remover arquivos 
RUN find ${BENCHMARK_DIR} -maxdepth 1 \( -name "*.TBL" -o -name "*.DBL" -o -name "RRTMG*" \) -exec rm -f {} \;
WORKDIR ${BENCHMARK_DIR}
RUN sed -i "s/config_run_duration = '3_00:00:00'/config_run_duration = '0_03:00:00'/g" namelist.atmosphere

#Linkar arquivos do modelo
RUN bash -c "\
    cd ${BENCHMARK_DIR} && \
    cp ../MPAS-Model_v8.2.2_tempohpc/docker/nvhpc_24.9/run_mpas.sh . && \
    for file in CAM_ABS_DATA.DBL CAM_AEROPT_DATA.DBL GENPARM.TBL LANDUSE.TBL NoahmpTable.TBL \
                OZONE_DAT.DBL OZONE_LAT.TBL OZONE_DAT.TBL OZONE_PLEV.TBL OZONE_TBL \
                RRTMG_LW_DATA RRTMG_LW_DATA.DBL RRTMG_SW_DATA RRTMG_SW_DATA.DBL \
                SOILPARM.TBL VEGPARM.TBL atmosphere_model; do \
        if [ -e ${MPAS_DIR}/\$file ]; then \
            ln -sf ${MPAS_DIR}/\$file .; \
        else \
            echo \"não encontrado\"; \
        fi; \
    done \
"

WORKDIR ${BENCHMARK_DIR}
ENTRYPOINT ["/bin/bash"]

run_mpas.sh 

In [ ]:
ntasks=${1}
nthreads=${2}

source /usr/share/modules/init/bash
module use /opt/nvidia/hpc_sdk/modulefiles
module load nvhpc-openmpi3/24.9

workdir=/home/monan/spack-0.23.1
spackdir=${workdir}
source ${spackdir}/share/spack/setup-env.sh

export SPACK_USER_CONFIG_PATH=${workdir}/.spack/${version}

export NETCDF=$(spack location -i netcdf-fortran)
export PNETCDF=$(spack location -i parallel-netcdf)

echo "NETCDF: ${NETCDF}"
echo "PNETCDF: ${PNETCDF}"

export LD_LIBRARY_PATH=$NETCDF/lib:$PNETCDF/lib:$LD_LIBRARY_PATH

echo $LD_LIBRARY_PATH

export OMP_NUM_THREADS=${nthreads}

mpirun -n ${ntasks} \
        --mca mpi_cuda_support 0 \
        ./atmosphere_model 2>&1 | tee run_mpas${ntasks}.out

No local do arquivo no terminal, execute o comando abaixo parar criar a imagem a partir do script MPAS_v8.2.2.dockerfile:

In [ ]:
docker build -t mpas:8.2.2 -f MPAS_v8.2.2.dockerfile .

Caso tenha feito tentativas anteriores que resultaram em erro, utilize a opção **--no-cache** para forçar a criação da imagem do zero, ignorando qualquer cache de etapas anteriores

In [ ]:
docker build --no-cache -t mpas:8.2.2 -f MPAS_v8.2.2.dockerfile .

Para visualizar a imagem criada utilize: 

In [ ]:
docker images

Aparecerá algo como:

In [ ]:
REPOSITORY   TAG        IMAGE ID       CREATED       SIZE
mpas         8.2.2      6c523f9c83ee   2 days ago    15.2GB
mpas         8.2.2-v3   77ab70c7690d   2 days ago    15.2GB
<none>       <none>     2b5b3d82a4a6   13 days ago   15.2GB

A imagem criada pode ser identificada pelo nome e versão (REPOSITORY e TAG) que definimos com a opção -t. No exemplo abaixo, o nome da imagem é **mpas** e a tag (versão) é **8.2.2**:


docker build --no-cache -t **mpas:8.2.2** -f MPAS_v8.2.2.dockerfile .

Após a criação da imagem, utilizamos o comando **docker run** para instanciar, ou seja, iniciar um container baseado nessa imagem:

In [ ]:
docker run -it --entrypoint bash mpas:8.2.2

Ao executar este comando, entramos no container no diretório **/home/monan/MPAS-A_benchmark_120km_v7.0** 
Para conferir os arquivos presentes, utilize o comando ls -ltr

In [ ]:
ls -ltr 

Precisa aparecer os seguintes aquivos:

Após isso podemos fazer a execução do mpas

In [ ]:
source ./run_mpas.sh 1 1

Saída esperada:

Para sair do container:

In [ ]:
exit

Ao usar o comando **exit**, você sai do container, mas ele não é removido — ele apenas é interrompido e entra no estado exited (parado).
Isso significa que ele ainda existe e pode ser acessado novamente a qualquer momento.
Para visualizar os containers existentes (em execução ou parados), utilize o comando abaixo:

In [ ]:
docker ps -a #visualizar containers existentes

In [ ]:
CONTAINER ID   IMAGE           COMMAND   CREATED       STATUS                       PORTS     NAMES
cb292ef2cdfe   mpas:8.2.2      "bash"    2 days ago    Exited (255) 8 minutes ago             vigorous_wiles
b9f80a1308ac   mpas:8.2.2-v3   "bash"    2 days ago    Exited (127) 2 days ago                objective_jones
0c5d0684fa02   2b5b3d82a4a6    "bash"    13 days ago   Exited (255) 10 days ago               vigilant_ganguly

Para executarmos o container novamente, utilizamos o comando:

In [ ]:
docker exec -i -t <container_name> bash

In [ ]:
#exemplo:
docker exec -i -t vigorous_wiles bash

obs.: o nome dos containers são gerados aleatóriamente

Também é possivel visualizar apenas containers que estão em execução:

In [ ]:
docker ps